# whisper-tools: A Deep Dive

**whisper-tools** is a small wrapper around [faster-whisper](https://github.com/SYSTRAN/faster-whisper) and OpenAI-compatible APIs for transcribing audio files, raw audio, and microphone streams. It provides a unified, typed interface across four backends:

- **Local transcription** — runs Whisper models on your own hardware via `faster-whisper`
- **API transcription** — sends audio to an OpenAI-compatible endpoint
- **Streaming** — captures microphone audio, detects speech, and transcribes in near-real-time
- **Diarization** — splits audio into speaker turns using `pyannote.audio`

This notebook walks through the architecture, the lazy-import design, the streaming flow, the typed result objects, scaling considerations, the CLI, and configuration files.

## 1. Architecture Overview

The library is organized into focused modules, each responsible for one concern:

| Module | Responsibility |
|--------|----------------|
| `whisper_tools/__init__.py` | Public API surface — re-exports the main classes and types |
| `whisper_tools/local.py` | `WhisperLocal` — local inference via `faster-whisper` |
| `whisper_tools/api.py` | `WhisperAPI` — remote inference via an OpenAI-compatible API |
| `whisper_tools/stream.py` | `StreamRecorder` — microphone capture + speech detection + transcription |
| `whisper_tools/diarize.py` | `Diarizer` — speaker diarization via `pyannote.audio` |
| `whisper_tools/audio.py` | Audio I/O helpers — loading, resampling, chunking, noise reduction |
| `whisper_tools/types.py` | Typed dataclasses — `Segment`, `TranscriptionResult`, `SpeakerTurn` |
| `whisper_tools/cli.py` | Command-line interface |
| `whisper_tools/config.py` | Configuration file loading (JSON / YAML) |

### The unified interface

Both `WhisperLocal` and `WhisperAPI` expose the same core methods:

- `transcribe(audio, sample_rate=16000, prompt=None, word_timestamps=False)` — accepts a file path **or** a raw numpy array
- `transcribe_many(audio_files, num_workers=...)` — parallel transcription of multiple files
- `WhisperAPI` additionally provides `transcribe_async(...)` for asyncio-based workflows

This means you can swap between local and API backends without changing your application code.

## 2. Lazy Imports: Heavy Dependencies Loaded Only When Used

One of the key design decisions in whisper-tools is **lazy importing**. Heavy dependencies — `faster-whisper`, `torch`, `pyannote.audio`, `sounddevice`, `openai`, `yaml` — are **not** imported at module load time. Instead, they are imported inside the methods that actually need them.

This has several benefits:

- **Fast startup** — importing `whisper_tools` does not pull in PyTorch or other heavy packages
- **Minimal footprint** — you can use the API backend without ever installing `faster-whisper`
- **Graceful degradation** — missing optional dependencies only raise errors when you try to use the corresponding feature

Let's verify this by checking what gets imported when we load the package:

In [ ]:
import sys

# Record which heavy modules are already in sys.modules
heavy_modules = ["faster_whisper", "torch", "pyannote", "sounddevice", "openai", "yaml"]
before = {m: (m in sys.modules) for m in heavy_modules}

after = {m: (m in sys.modules) for m in heavy_modules}

print("Heavy modules loaded after `import whisper_tools`:")
for m in heavy_modules:
    status = "LOADED" if after[m] else "not loaded"
    print(f"  {m:20s} -> {status}")

### Where the lazy imports happen

Here's a map of where each heavy dependency gets loaded:

| Dependency | Loaded in | When |
|------------|-----------|------|
| `faster_whisper.WhisperModel` | `WhisperLocal._load()` | First call to `transcribe()` |
| `torch` | `WhisperLocal._cuda_available()` / `Diarizer.diarize()` | First device check or diarization call |
| `pyannote.audio.Pipeline` | `Diarizer._load()` | First call to `diarize()` |
| `sounddevice` | `StreamRecorder._record()` / `list_devices()` | First streaming session |
| `openai.OpenAI` / `AsyncOpenAI` | `WhisperAPI.__init__()` / `_get_async_client()` | When the API client is constructed |
| `yaml` | `load_config()` | Only when reading a `.yaml`/`.yml` config file |

The `__init__.py` file only imports the lightweight public classes and types:

In [ ]:
import inspect
import whisper_tools

print("Public API:")
for name in whisper_tools.__all__:
    obj = getattr(whisper_tools, name)
    kind = "class" if inspect.isclass(obj) else "function"
    print(f"  {name:20s} -> {kind}")

print(f"\nVersion: {whisper_tools.__version__}")

## 3. Local Transcription (`WhisperLocal`)

`WhisperLocal` runs Whisper models **on your own machine** using `faster-whisper`, which is a CTranslate2-based reimplementation that is significantly faster and more memory-efficient than the original OpenAI implementation.

### Key parameters

- `model` — model name or path (e.g. `"base"`, `"small"`, `"medium"`, `"large-v3"`, or a local path)
- `language` — language code (default `"ru"`); set to `None` for auto-detection
- `device` — `"auto"`, `"cpu"`, `"cuda"`, or `"mps"`
- `chunk_seconds` — chunk length for long files (default 30.0)
- `overlap_seconds` — overlap between chunks (default 2.0)
- `noise_reduction` — spectral-gating noise reduction strength (0 disables)

### How long files are handled

When the audio is longer than `chunk_seconds`, `WhisperLocal` splits it into overlapping chunks, transcribes each chunk independently, and merges the results. The overlap prevents words from being cut off at chunk boundaries.

In [ ]:
from whisper_tools import WhisperLocal

# Create a local transcriber with the 'base' model, Russian language
whisper = WhisperLocal(
    model="base",           # model size: tiny/base/small/medium/large-v3
    language="ru",          # language code, or None for auto-detection
    device="auto",          # auto/cpu/cuda/mps
    chunk_seconds=30.0,     # chunk length for long files
    overlap_seconds=2.0,    # overlap between chunks
    noise_reduction=0.0,    # 0 disables noise reduction
)

# Transcribe a file (or pass a numpy array directly)
# result = whisper.transcribe("audio.wav")

# For this demo, let's create a synthetic audio array and transcribe it
import numpy as np

# Generate 3 seconds of silence (no speech) — the model will return empty text
sample_rate = 16000
silence = np.zeros(sample_rate * 3, dtype=np.float32)

# Uncomment to actually run:
# result = whisper.transcribe(silence, sample_rate=sample_rate)
# print(result.text)

print("WhisperLocal created. The model is loaded lazily on first transcribe() call.")
print(f"  model={whisper.model_name}, language={whisper.language}, device={whisper.device}")

In [ ]:
# Word-level timestamps
# result = whisper.transcribe("audio.wav", word_timestamps=True)
# for segment in result.segments:
#     print(f"[{segment.start:.1f}-{segment.end:.1f}] {segment.text}")
#     if segment.words:
#         for start, end, word in segment.words:
#             print(f"    {word} [{start:.2f}-{end:.2f}]")

print("Word timestamps are available when word_timestamps=True is passed.")
print("Each Segment.words is a list of (start, end, word) tuples.")

## 4. API Transcription (`WhisperAPI`)

`WhisperAPI` sends audio to an **OpenAI-compatible** transcription endpoint. This is useful when you want to offload computation to a server, use a hosted model, or avoid installing heavy local dependencies.

### Key parameters

- `api_key` — your API key
- `base_url` — the API base URL (defaults to OpenAI's endpoint)
- `model` — model name (default `"whisper-large-v3"`)
- `language` — language code (default `"ru"`); `None` for auto-detection
- `max_retries` — number of retries for transient failures (default 2)
- `timeout` — request timeout in seconds (default 60.0)
- `noise_reduction` — noise reduction strength (0 disables)

### How it works

1. Audio is prepared (loaded, resampled to 16 kHz, optionally noise-reduced)
2. If the input is a numpy array, it's saved to a temporary WAV file
3. The file is sent to the API with `response_format="verbose_json"`
4. The response is parsed into a `TranscriptionResult`
5. The temporary file is cleaned up

The API client supports both **sync** (`transcribe`) and **async** (`transcribe_async`) calls, and **parallel** transcription via `transcribe_many`.

In [ ]:
from whisper_tools import WhisperAPI

# Create an API transcriber
api = WhisperAPI(
    api_key="your-api-key",
    base_url="https://api.openai.com/v1", 
    model="whisper-large-v3",
    language="ru",
    max_retries=2,      
    timeout=60.0,      
)

# Transcribe a file
# result = api.transcribe("audio.wav")
# print(result.text)

# Async version
# import asyncio
# async def main():
#     result = await api.transcribe_async("audio.wav")
#     print(result.text)
# asyncio.run(main())

print("WhisperAPI created. The OpenAI client is constructed eagerly in __init__.")
print(f"  model={api.model}, base_url={api.base_url}")

In [ ]:
# Parallel transcription with transcribe_many
# files = ["audio1.wav", "audio2.wav", "audio3.wav"]
# results = api.transcribe_many(files, num_workers=4)
# for path, result in zip(files, results):
#     print(f"{path}: {result.text}")

print("transcribe_many uses a ThreadPoolExecutor to process files in parallel.")
print("The OpenAI client is thread-safe, so a single instance can be shared.")

## 5. Streaming (`StreamRecorder`)

`StreamRecorder` provides **near-real-time transcription** from the microphone. It combines three stages:

```
Audio capture → Speech detection → Transcription
```

### The streaming flow

1. **Audio capture** — a background thread opens a `sounddevice.InputStream` and pushes audio blocks into a thread-safe queue
2. **Speech detection** — `_next_chunk()` pulls audio from the queue and checks two thresholds:
   - `energy_threshold` — mean squared energy must exceed this value
   - `dynamic_threshold` — peak-to-peak amplitude must exceed this value
3. **Adaptive chunking** — the chunk length grows when speech is detected (up to `max_chunk_seconds`) and shrinks during silence (down to 2.0s). This keeps transcription responsive during short utterances and captures longer phrases without cutting them off.
4. **Transcription** — the chunk is passed to the transcriber (any object with a `transcribe(audio, sample_rate)` method)
5. **Finalization** — results ending with sentence punctuation (`.`, `!`, `?`, `…`) are considered **finalized**. Partial results are merged with the next finalized result.

### Two usage modes

- **Push-based** — pass `on_text=lambda r: print(r.text)` and the recorder delivers results automatically in a background thread
- **Pull-based** — call `recorder.process()` manually in a loop to get the next result (or `None` if no speech yet)

In [ ]:
from whisper_tools import list_devices

devices = list_devices()
print("Available input devices:")
for index, name in devices:
    print(f"  [{index}] {name}")

In [ ]:
# Streaming example — push-based (callback) mode
from whisper_tools import WhisperLocal, StreamRecorder

# Create a local transcriber (or use WhisperAPI for remote transcription)
transcriber = WhisperLocal(model="base", language="ru")

# Create a StreamRecorder with a callback that prints each result
recorder = StreamRecorder(
    transcriber,
    sample_rate=16000,
    chunk_seconds=2.1,       # initial chunk length
    max_chunk_seconds=5.0,   # upper bound for adaptive growth
    adapt_step=0.5,          # how much the chunk grows/shrinks per adaptation
    min_interval=1.0,        # minimum time between transcription attempts
    energy_threshold=1e-5,   # speech detection: mean squared energy
    dynamic_threshold=5e-3,  # speech detection: peak-to-peak amplitude
    max_queue_size=20,       # if exceeded, oldest blocks are dropped
    on_text=lambda r: print(r.text, flush=True),
)

# Start recording
# recorder.start()
# try:
#     input("Press Enter to stop\n")
# finally:
#     recorder.stop()

print("StreamRecorder created. Call start() to begin microphone capture.")
print("Results are delivered via the on_text callback in a background thread.")

In [ ]:
# Streaming example — pull-based (polling) mode
recorder = StreamRecorder(WhisperLocal(model="base", language="ru"))

# recorder.start()
# try:
#     while True:
#         result = recorder.process()  # returns None if no speech yet
#         if result:
#             print(result.text)
# except KeyboardInterrupt:
#     pass
# finally:
#     recorder.stop()

print("Pull-based mode: call recorder.process() in a loop to get results.")
print("process() returns None when there is no speech in the current chunk.")

### Streaming internals

The `StreamRecorder` uses several internal mechanisms:

- **Thread-safe queue** — audio blocks from the microphone callback are pushed into a `queue.Queue`
- **Background threads** — one for audio capture (`_record`), one for result delivery (`_push_loop`)
- **Lag detection** — if the queue exceeds `max_queue_size`, the oldest blocks are dropped and `recorder.lagging` is set to `True`. This indicates transcription is slower than real-time.
- **Partial result merging** — when a chunk ends mid-sentence (no terminal punctuation), the result is stored as `_pending`. The next finalized result is merged with it, producing a complete utterance.

The `finalized` flag on `TranscriptionResult` distinguishes partial hypotheses from complete utterances.

## 6. Speaker Diarization (`Diarizer`)

`Diarizer` uses `pyannote.audio` to split audio into **speaker turns** — segments of audio attributed to a specific speaker. This is useful for meeting transcription, interview analysis, and any scenario where you need to know "who said what."

### Requirements

- Install the optional extra: `pip install whisper-tools[diarize]`
- A Hugging Face access token with access to the `pyannote/speaker-diarization-3.1` model

### How it works

1. Audio is loaded and resampled to 16 kHz
2. The `pyannote.audio` pipeline is loaded (lazily, on first use)
3. The waveform is passed through the pipeline
4. The resulting speaker turns are returned as a list of `SpeakerTurn` objects

In [ ]:
from whisper_tools import Diarizer

diarizer = Diarizer(
    hf_token="hf_...",  # Hugging Face token with access to the model
    model="pyannote/speaker-diarization-3.1",
)

# turns = diarizer.diarize("audio.wav")
# for turn in turns:
#     print(f"{turn.speaker}: {turn.start:.1f}-{turn.end:.1f}")

# Convenience function for one-off use
# turns = diarize("audio.wav", hf_token="hf_...")

print("Diarizer created. The pyannote pipeline is loaded lazily on first diarize() call.")
print("The convenience function diarize() creates a Diarizer and calls diarize() in one step.")

## 7. Typed Results

All transcription and diarization methods return **typed dataclasses**, making the results self-documenting and IDE-friendly.

### `Segment`

A timed piece of transcribed text.

| Field | Type | Description |
|-------|------|-------------|
| `start` | `float` | Start time in seconds |
| `end` | `float` | End time in seconds |
| `text` | `str` | Transcribed text |
| `words` | `list[(float, float, str)] \| None` | Word-level timestamps when requested |

### `TranscriptionResult`

The result of a transcription call.

| Field | Type | Description |
|-------|------|-------------|
| `text` | `str` | Full transcribed text |
| `segments` | `list[Segment]` | Timed segments |
| `language` | `str \| None` | Detected or requested language code |
| `duration` | `float` | Audio duration in seconds |
| `finalized` | `bool` | Whether the text is a complete utterance (set by `StreamRecorder`) |

### `SpeakerTurn`

A speaker turn from diarization.

| Field | Type | Description |
|-------|------|-------------|
| `start` | `float` | Start time in seconds |
| `end` | `float` | End time in seconds |
| `speaker` | `str` | Speaker label |

In [ ]:
from whisper_tools import Segment, TranscriptionResult, SpeakerTurn
from dataclasses import fields

for cls in [Segment, TranscriptionResult, SpeakerTurn]:
    print(f"{cls.__name__}:")
    for f in fields(cls):
        print(f"  {f.name:12s} : {f.type}")
    print()

In [ ]:
from whisper_tools import Segment, TranscriptionResult, SpeakerTurn

# Build a TranscriptionResult from segments
segments = [
    Segment(start=0.0, end=2.5, text="Привет, мир!"),
    Segment(start=2.5, end=5.0, text="Как дела?"),
]

result = TranscriptionResult(
    text="Привет, мир! Как дела?",
    segments=segments,
    language="ru",
    duration=5.0,
    finalized=True,
)

print(f"Text: {result.text}")
print(f"Language: {result.language}")
print(f"Duration: {result.duration:.1f}s")
print(f"Segments: {len(result.segments)}")
for seg in result.segments:
    print(f"  [{seg.start:.1f}-{seg.end:.1f}] {seg.text}")

# Speaker turns
turns = [
    SpeakerTurn(start=0.0, end=3.0, speaker="SPEAKER_00"),
    SpeakerTurn(start=3.0, end=6.0, speaker="SPEAKER_01"),
]
print("\nSpeaker turns:")
for turn in turns:
    print(f"  {turn.speaker}: {turn.start:.1f}-{turn.end:.1f}")

## 8. Scaling

### Model sizes

Whisper models come in several sizes, trading accuracy for speed and memory:

| Model | Parameters | VRAM (fp16) | Speed | Accuracy |
|-------|-----------|-------------|-------|----------|
| `tiny` | 39M | ~1 GB | Fastest | Lowest |
| `base` | 74M | ~1 GB | Very fast | Low |
| `small` | 244M | ~2 GB | Fast | Medium |
| `medium` | 769M | ~5 GB | Moderate | High |
| `large-v3` | 1550M | ~10 GB | Slow | Highest |

### GPU vs CPU

- **GPU (CUDA)** — `device="cuda"` uses `float16` compute type, which is significantly faster than CPU
- **CPU** — `device="cpu"` uses `int8` quantization, which is much faster than fp32 on CPU
- **Apple Silicon (MPS)** — `device="mps"` uses `float16` on the Metal Performance Shaders backend
- **Auto** — `device="auto"` checks for CUDA availability and falls back to CPU

### Parallel transcription

Both `WhisperLocal` and `WhisperAPI` support `transcribe_many(files, num_workers=N)` for parallel processing:

- **`WhisperLocal`** — each worker gets its **own** model instance, because a single `faster-whisper` model is not thread-safe. This means `num_workers` models are loaded into memory simultaneously.
- **`WhisperAPI`** — a single OpenAI client is thread-safe, so one client is shared across all workers.

### Practical scaling tips

1. **Start with `base`** for development, then scale up to `small`/`medium` for production accuracy
2. **Use GPU when available** — the speedup is dramatic, especially for larger models
3. **For batch jobs**, use `transcribe_many` with `num_workers` matching your CPU/GPU cores
4. **For long files**, `WhisperLocal` automatically chunks them with overlap to avoid memory issues
5. **For streaming**, use a small model (`base`) to keep up with real-time audio

In [ ]:
# Transcribe multiple files in parallel
# files = ["audio1.wav", "audio2.wav", "audio3.wav", "audio4.wav"]
# whisper = WhisperLocal(model="base", language="ru")
# results = whisper.transcribe_many(files, num_workers=4)

# for path, result in zip(files, results):
#     print(f"{path}: {result.text}")

print("transcribe_many creates a separate WhisperLocal instance per worker.")
print("This is because faster-whisper models are not thread-safe.")
print("num_workers controls how many models are loaded simultaneously.")

In [ ]:
# Device selection example
from whisper_tools import WhisperLocal

# CPU with int8 quantization
cpu_whisper = WhisperLocal(model="base", device="cpu")

# GPU with float16
# gpu_whisper = WhisperLocal(model="base", device="cuda")

# Apple Silicon
# mps_whisper = WhisperLocal(model="base", device="mps")

# Auto-detect (CUDA if available, else CPU)
auto_whisper = WhisperLocal(model="base", device="auto")

print(f"CPU device: {cpu_whisper.device}")
print(f"Auto device: {auto_whisper.device}")
print("\nCompute types:")
print("  cuda/mps -> float16")
print("  cpu      -> int8")

## 9. CLI Usage

whisper-tools ships with a command-line interface (`whisper-tools`) that wraps the Python API. It supports local transcription, API transcription, streaming, and device listing.

### Basic commands

```bash
# Transcribe a file (uses local WhisperLocal by default)
whisper-tools audio.wav

# Output as JSON
whisper-tools --json audio.wav

# Stream from the microphone
whisper-tools --stream

# List available input devices
whisper-tools --list-devices

# Show version
whisper-tools --version
```

### Options

| Option | Description |
|--------|-------------|
| `files` | Audio files to transcribe (positional) |
| `--stream` | Stream from the microphone |
| `--list-devices` | List input devices and exit |
| `--version` | Print version and exit |
| `--config PATH` | Path to a JSON or YAML config file |
| `--model NAME` | Model name for local transcription (default: base) |
| `--language CODE` | Language code, or `auto` for auto-detection (default: ru) |
| `--device DEVICE` | Device for local transcription (default: auto) |
| `--api-key KEY` | API key; when set, uses the API backend |
| `--base-url URL` | API base URL (default: OpenAI) |
| `--api-model NAME` | API model name (default: whisper-large-v3) |
| `--noise-reduction FLOAT` | Noise reduction strength (default: 0) |
| `--device-index INT` | Input device index for streaming |
| `-o, --output FILE` | Write results to a file instead of stdout |
| `--json` | Output results as JSON |

### Backend selection

- **Local** (default) — uses `WhisperLocal` with the specified `--model`
- **API** — set `--api-key` to switch to `WhisperAPI`; optionally set `--base-url` and `--api-model`

In [ ]:
# CLI examples (run in a terminal, not in this notebook)
#
# Transcribe a file locally
# whisper-tools audio.wav
#
# Transcribe with a specific model and language
# whisper-tools --model small --language en audio.wav
#
# Use the API backend
# whisper-tools --api-key sk-... --base-url https://api.openai.com/v1 audio.wav
#
# Output JSON to a file
# whisper-tools --json -o result.json audio.wav
#
# Stream from the microphone with a specific device
# whisper-tools --stream --device-index 0
#
# Use a config file
# whisper-tools --config whisper-tools.yaml audio.wav

print("Run these commands in your terminal.")
print("The CLI uses the same WhisperLocal/WhisperAPI classes under the hood.")

## 10. Configuration Files

whisper-tools supports **JSON** and **YAML** configuration files. The CLI looks for `whisper-tools.json`, `whisper-tools.yaml`, or `whisper-tools.yml` in the current directory, or you can specify a path with `--config`.

### JSON example

```json
{
  "model": "small",
  "language": "ru",
  "device": "auto",
  "noise_reduction": 0.3,
  "api_key": "sk-...",
  "base_url": "https://api.openai.com/v1",
  "api_model": "whisper-large-v3"
}
```

### YAML example

```yaml
model: small
language: ru
device: auto
noise_reduction: 0.3
api_key: sk-...
base_url: https://api.openai.com/v1
api_model: whisper-large-v3
```

### Precedence

Command-line arguments **override** config file values. The merge order is:

1. Defaults (hardcoded in `cli.py`)
2. Config file values
3. Command-line arguments

### How config loading works

The `load_config()` function in `whisper_tools/config.py`:

1. If a `path` is given, reads that file (JSON or YAML based on extension)
2. Otherwise, looks for `whisper-tools.json`, `whisper-tools.yaml`, `whisper-tools.yml` in the current directory
3. Returns an empty dict if no file is found

YAML support requires the optional `pyyaml` dependency (`pip install whisper-tools[yaml]`).

In [ ]:
from whisper_tools.config import load_config

# Load from a specific file
# config = load_config("whisper-tools.yaml")

# Or let it auto-discover whisper-tools.json/yaml/yml in the current directory
config = load_config()

print("Loaded config:")
print(config if config else "(no config file found — using defaults)")

In [ ]:
# write a config file and load it
import json
import tempfile
import os

# Create a sample config
sample_config = {
    "model": "small",
    "language": "ru",
    "device": "auto",
    "noise_reduction": 0.3,
    "api_key": "sk-...",
    "base_url": "https://api.openai.com/v1",
    "api_model": "whisper-large-v3",
}

# Write to a temp file
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(sample_config, f, indent=2)
    temp_path = f.name

# Load it back
loaded = load_config(temp_path)
print("Loaded config from temp file:")
print(json.dumps(loaded, indent=2))

# Clean up
os.unlink(temp_path)

## Summary

whisper-tools provides a clean, unified interface for speech-to-text across four backends:

1. **Local transcription** (`WhisperLocal`) — fast, private, on-device inference via `faster-whisper`
2. **API transcription** (`WhisperAPI`) — remote inference via OpenAI-compatible endpoints
3. **Streaming** (`StreamRecorder`) — real-time microphone transcription with adaptive chunking and speech detection
4. **Diarization** (`Diarizer`) — speaker attribution via `pyannote.audio`

Key design principles:

- **Lazy imports** — heavy dependencies are loaded only when needed
- **Typed results** — `TranscriptionResult`, `Segment`, and `SpeakerTurn` dataclasses
- **Unified interface** — swap between local and API backends without changing code
- **Parallel processing** — `transcribe_many` for batch jobs
- **CLI + config files** — scriptable and configurable from the command line